In [1]:
#shapely 2.0 版本建议使用下面注释的1行
import os
#os.environ['USE_PYGEOS'] = '0'
import geopandas as gp
import pandas as pd
import shapely

In [17]:
#使用geopandas读取路网文件，路网文件需要先进行前处理，比如进行相交线打断操作推荐使用arcgis
#需要转换成投影坐标计算长度
roads = gp.read_file("data\street_map\sydney_roads_graph.shp")
roads = roads.to_crs(epsg=32756)
roads = roads[roads.geometry.type == 'LineString']
roads['length'] = roads.length
roads = roads.to_crs(epsg=4326)

In [18]:
#以下的转换代码主要为获取线的两个端点，并重新编码，匹配后可获得图结构的顶点和边情况
# Compute the start- and end-position based on linestring
roads['Start_pos'] = roads.geometry.apply(lambda x: x.coords[0])
roads['End_pos'] = roads.geometry.apply(lambda x: x.coords[-1])
roads.head(5)

,osm_id,code,fclass,name,ref,oneway,maxspeed,layer,bridge,tunnel,length,geometry,Start_pos,End_pos
0,27007114,5114,secondary,Sir Bertram Stevens Drive,None,B,60,0,F,F,448.285949,"LINESTRING (151.05010 -34.17040, 151.04990 -34...","(151.05010029999994, -34.170397099999946)","(151.04536545130816, -34.17037752630119)"
1,27007114,5114,secondary,Sir Bertram Stevens Drive,None,B,60,0,F,F,285.636851,"LINESTRING (151.04265 -34.17004, 151.04250 -34...","(151.0426535966933, -34.170039336263635)","(151.0396759197206, -34.16966797263269)"
2,329309047,5115,tertiary,Garie Road,None,B,0,0,F,F,1130.775761,"LINESTRING (151.05010 -34.17040, 151.05027 -34...","(151.05010029999994, -34.170397099999946)","(151.05975443900138, -34.168349397344905)"
3,329309047,5115,tertiary,Garie Road,None,B,0,0,F,F,1468.308139,"LINESTRING (151.06079 -34.16880, 151.06084 -34...","(151.0607861038004, -34.16879696536892)","(151.0665583000001, -34.16994590000001)"
4,27007142,5114,secondary,Lady Wakehurst Drive,None,B,70,0,F,F,1073.995674,"LINESTRING (151.01839 -34.15767, 151.01810 -34...","(151.01839230000007, -34.15766609999996)","(151.01672664141586, -34.16654045926015)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145240,172652393,5115,tertiary,Pitt Town Road,None,B,70,0,F,F,1614.906922,"LINESTRING (150.94308 -33.60796, 150.94266 -33...","(150.94307589999994, -33.60796270000004)","(150.93023390000008, -33.5993851)"
145241,201202429,5115,tertiary,Pitt Town Road,None,B,60,1,T,F,7.659221,"LINESTRING (150.93015 -33.59937, 150.93023 -33...","(150.9301528191478, -33.59937227866185)","(150.93023390000008, -33.5993851)"
145242,25822617,5114,secondary,Loop Road,None,F,0,0,F,F,3.157208,"LINESTRING (151.09717 -33.82238, 151.09720 -33...","(151.09717413892764, -33.82237617747647)","(151.09719530000007, -33.82239850000002)"
145243,5066695,5113,primary,Burns Bay Road,None,F,70,1,T,F,5.594874,"LINESTRING (151.14659 -33.84069, 151.14663 -33...","(151.14658927135818, -33.840689387948366)","(151.14662700000008, -33.84072880000002)"


In [19]:
# Create Series of unique nodes and their associated position
s_points = pd.concat([roads.Start_pos,roads.End_pos], ignore_index=True)
s_points = s_points.drop_duplicates().reset_index(drop=True)
s_points

0         (151.05010029999994, -34.170397099999946)
1          (151.0426535966933, -34.170039336263635)
2           (151.0607861038004, -34.16879696536892)
3          (151.01839230000007, -34.15766609999996)
4          (151.02882450000004, -34.15224389999997)
                            ...                    
113984            (150.93023390000008, -33.5993851)
113985     (151.09719530000007, -33.82239850000002)
113986     (151.14662700000008, -33.84072880000002)
113987    (151.14510033566114, -33.837745419566595)
113988     (151.21197441895345, -33.85052262255607)
Length: 113989, dtype: object

In [20]:
# Add index of start and end node of linestring to geopandas DataFrame
df_points = pd.DataFrame(s_points, columns=['Start_pos'])
df_points['FNODE_'] = df_points.index
roads = pd.merge(roads, df_points, on='Start_pos', how='inner')
df_points = pd.DataFrame(s_points, columns=['End_pos'])
df_points['TNODE_'] = df_points.index
roads = pd.merge(roads, df_points, on='End_pos', how='inner')
roads.head()

,osm_id,code,fclass,name,ref,oneway,maxspeed,layer,bridge,tunnel,length,geometry,Start_pos,End_pos,FNODE_,TNODE_
0,27007114,5114,secondary,Sir Bertram Stevens Drive,None,B,60,0,F,F,448.285949,"LINESTRING (151.05010 -34.17040, 151.04990 -34...","(151.05010029999994, -34.170397099999946)","(151.04536545130816, -34.17037752630119)",0,102072
1,329309047,5115,tertiary,Garie Road,None,B,0,0,F,F,1130.775761,"LINESTRING (151.05010 -34.17040, 151.05027 -34...","(151.05010029999994, -34.170397099999946)","(151.05975443900138, -34.168349397344905)",0,102074
2,27007114,5114,secondary,Sir Bertram Stevens Drive,None,B,60,0,F,F,285.636851,"LINESTRING (151.04265 -34.17004, 151.04250 -34...","(151.0426535966933, -34.170039336263635)","(151.0396759197206, -34.16966797263269)",1,102073
3,329309047,5115,tertiary,Garie Road,None,B,0,0,F,F,1468.308139,"LINESTRING (151.06079 -34.16880, 151.06084 -34...","(151.0607861038004, -34.16879696536892)","(151.0665583000001, -34.16994590000001)",2,102075
4,27007142,5114,secondary,Lady Wakehurst Drive,None,B,70,0,F,F,1073.995674,"LINESTRING (151.01839 -34.15767, 151.01810 -34...","(151.01839230000007, -34.15766609999996)","(151.01672664141586, -34.16654045926015)",3,102076


In [21]:
# Bring nodes and their position in form needed for osmnx (give arbitrary osmid (index) despite not osm file)
df_points.columns = ['pos', 'osmid'] 
df_points[['x', 'y']] = df_points['pos'].apply(pd.Series)
df_node_xy = df_points.drop('pos', axis=1)
df_node_xy

,osmid,x,y
0,0,151.050100,-34.170397
1,1,151.042654,-34.170039
2,2,151.060786,-34.168797
3,3,151.018392,-34.157666
4,4,151.028825,-34.152244
...,...,...,...
113984,113984,150.930234,-33.599385
113985,113985,151.097195,-33.822399
113986,113986,151.146627,-33.840729
113987,113987,151.145100,-33.837745


In [22]:
import igraph as ig
G = ig.Graph()
G.add_vertices(len(s_points))
G.add_edges(roads[['FNODE_','TNODE_']].values)


In [23]:
G.vs["id"] = df_node_xy['osmid'].values
G.vs["x"] = df_node_xy['x'].values
G.vs["y"] = df_node_xy['y'].values
G.es["length"] = roads['length'].values

In [24]:
#广度优先遍历找到最大连通网络，阈值设置为超过总节点的90%
T_vs_num = len(df_node_xy)
bfs_vs_num = 0
init_point = 0
max_test_num = 100
while (bfs_vs_num/T_vs_num)<0.9 or init_point<T_vs_num:
    G_bfs= G.bfs(init_point)
    bfs_vs_num=len(G_bfs[0])
    init_point += int(T_vs_num/max_test_num)
print(len(G_bfs[0])/len(df_node_xy))

0.9882707980594618


In [25]:
roads = roads[roads.FNODE_.isin(G_bfs[0])|roads.TNODE_.isin(G_bfs[0])]

In [26]:
import math
def nearest_point(order_info,network_info):
	def geodistance(n1:set,n2:set):
		lng1, lat1, lng2, lat2 = map(math.radians, [float(n1[0]), float(n1[1]), float(n2[0]), float(n2[1])]) # 经纬度转换成弧度
		dlon=lng2-lng1
		dlat=lat2-lat1

		a=math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
		distance=2*math.asin(math.sqrt(a))*6371*1000 # 地球平均半径，6371km
		distance=round(distance,4)
		return distance
	
	multi_point = shapely.geometry.MultiPoint(network_info[['x','y']].values) # 将路网节点转为multipoint对象
	node_list=[]
	distance_list=[]
	for i in order_info[['longitude','latitude']].values:
	# 读取指定点坐标，并转为point对象
		point_coor = shapely.geometry.Point(i[0],i[1]) 
		
		# 求指定点到路网上的最近点
		nearest_point = shapely.ops.nearest_points(point_coor, multi_point)
		distance = geodistance(nearest_point[0].coords[0],nearest_point[1].coords[0])
		node_id = network_info[(network_info.x==nearest_point[1].x) & (network_info.y==nearest_point[1].y)].osmid.values[0]
		node_list.append(node_id)
		distance_list.append(distance)
	order_info['node_id']=node_list
	order_info['distance_to_node']=distance_list
	return 	order_info


In [27]:
order_filename = "data\point_data\point_data.csv"
order_gps_data = pd.read_csv(order_filename)
order_gps_data


,id,longitude,latitude
0,0,151.278807,-33.850326
1,1,150.847370,-34.002706
2,2,151.217132,-33.862889
3,3,150.721389,-34.002940
4,4,151.066590,-33.836043
5,5,150.867575,-33.892458
6,6,151.052392,-33.929684
7,7,151.268187,-33.731880
8,8,151.051032,-34.047222
9,9,150.906240,-33.773748


In [28]:
#将订单点数据匹配到最大连通网络上的节点，并计算两点距离
order_node_info = nearest_point(order_gps_data,df_node_xy[df_node_xy.osmid.isin(G_bfs[0])])
order_node_info

,id,longitude,latitude,node_id,distance_to_node
0,0,151.278807,-33.850326,51966,28.5356
1,1,150.847370,-34.002706,9064,327.6460
2,2,151.217132,-33.862889,48786,314.5377
3,3,150.721389,-34.002940,7850,1027.8615
4,4,151.066590,-33.836043,55043,123.8192
5,5,150.867575,-33.892458,37042,24.8322
6,6,151.052392,-33.929684,24517,44.8085
7,7,151.268187,-33.731880,88939,103.4812
8,8,151.051032,-34.047222,1933,94.0712
9,9,150.906240,-33.773748,73348,73.7608


In [31]:
#保证匹配节点距离不超过某个值，超过说明该点未能匹配上，可能是路网数据质量等问题
#因为这个数据我根据做了偏移，所以匹配出来有的距离偏差较大，最好用实际的数据
matrix_node = []
order_sample=order_node_info[order_node_info.distance_to_node<1200]
#对匹配后的路网节点进行去重，igraph距离矩阵函数不接收重复输入
[matrix_node.append(i) for i in order_sample['node_id'].values if not i in matrix_node]
distance_matrix = G.distances(matrix_node,matrix_node,weights =  G.es['length'])
order_sample.distance_to_node.max()

1027.8615

In [32]:
#防止出现找不到路出现无穷的情况，但是最大连通网络上不会有这种问题，可用可不用。
for i,a in enumerate(distance_matrix):
    if max(a) == float('inf'):
        for j,b in enumerate(a):
            if b == float('inf'):
                distance_matrix[i][j]=99999999999

In [33]:
#路径规划，Google的or-tools
"""Simple Travelling Salesperson Problem (TSP) on a circuit board."""

import math
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp


def create_data_model(distance_matrix):
    """Stores the data for the problem."""
    data = {}
    # Locations in block units
    data['locations'] = distance_matrix
    data['num_vehicles'] = 1
    data['depot'] = [0]
    data['end'] = [10]
    return data


def compute_euclidean_distance_matrix(locations):
    """Creates callback to return distance between points."""
    distances = {}
    for from_counter, from_node in enumerate(locations):
        distances[from_counter] = {}
        for to_counter, to_node in enumerate(locations):
            if from_counter == to_counter:
                distances[from_counter][to_counter] = 0
            else:
                # Euclidean distance
                distances[from_counter][to_counter] = (int(
                    math.hypot((from_node[0] - to_node[0]),
                               (from_node[1] - to_node[1]))))
    return distances


def print_solution(manager, routing, solution):
    """Prints solution on console."""
    print('Objective: {}'.format(solution.ObjectiveValue()))
    index = routing.Start(0)
    plan_output = 'Route:\n'
    route_distance = 0
    while not routing.IsEnd(index):
        plan_output += ' {} ->'.format(manager.IndexToNode(index))
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan_output += ' {}\n'.format(manager.IndexToNode(index))
    print(plan_output)
    plan_output += 'Objective: {}m\n'.format(route_distance)



"""Entry point of the program."""
# Instantiate the data problem.
data = create_data_model(distance_matrix)

# Create the routing index manager.
manager = pywrapcp.RoutingIndexManager(len(data['locations']),
                                        data['num_vehicles'], data['depot'],data['end'])

# Create Routing Model.
routing = pywrapcp.RoutingModel(manager)

distance_matrix = compute_euclidean_distance_matrix(data['locations'])

def distance_callback(from_index, to_index):
    """Returns the distance between the two nodes."""
    # Convert from routing variable Index to distance matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

# Define cost of each arc.
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Setting first solution heuristic.
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
search_parameters.time_limit.seconds = 300

# Solve the problem.
solution = routing.SolveWithParameters(search_parameters)

# Print solution on console.
if solution:
    print_solution(manager, routing, solution)

Objective: 218758
Route:
 0 -> 19 -> 7 -> 16 -> 13 -> 14 -> 3 -> 1 -> 5 -> 9 -> 8 -> 12 -> 6 -> 4 -> 15 -> 17 -> 18 -> 11 -> 2 -> 10



In [34]:
#获取路径节点顺序
def get_routes(solution, routing, manager):
  """Get vehicle routes from a solution and store them in an array."""
  # Get vehicle routes and store them in a two dimensional array whose
  # i,j entry is the jth location visited by vehicle i along its route.
  routes = []
  for route_nbr in range(routing.vehicles()):
    index = routing.Start(route_nbr)
    route = [manager.IndexToNode(index)]
    while not routing.IsEnd(index):
      index = solution.Value(routing.NextVar(index))
      route.append(manager.IndexToNode(index))
    routes.append(route)
  return routes
routes = get_routes(solution, routing, manager)

In [35]:
#生成简略图
import folium
#folium可能需要梯子才能显示地图
def folium_markers(m,gps_list):
    a = 0
    for i in gps_list:
        folium.Circle(
            location=[i[1],i[0]],
            radius=30,
            popup="Laurelhurst Park",
            color="#3186cc",
            fill=True,
            fill_color="#3186cc",
            
            ).add_to(m)
        a += 1
    return m
trail_coordinates  = []
for i in routes[0]:
    trail_coordinates.append(order_node_info[order_node_info.node_id==matrix_node[i]][['latitude','longitude']].values[0])

m = folium.Map(location=[-33.950195, 151.165009],zoom_start=11)
folium_markers(m,order_sample[['longitude','latitude']].values)
folium.PolyLine(trail_coordinates, tooltip="Coast").add_to(m)
m

In [36]:
#计算详细的最短路径，以获得实际路径显示
routes_trace=[]
n=0

for i in routes[0]:
    if n==0:
        from_node = matrix_node[i]
        n=1
    else :
        to_node = matrix_node[i]
        routes_trace.append(G.get_shortest_paths(from_node,to_node)[0])
        from_node = to_node

In [37]:
#从详细最短路径中获取相关线数据，用于显示，应该要考虑回程去重，这里没有做，效率较低
#无向图没有方向，所以from_to不是固定的，只要在两点均在集合里就认为是这条线
n=0
for route in routes_trace:
    for i in range(len(route)-1):
        from_to_set = (route[i],route[i+1])

        if n == 0:
            routes_trace_df=roads[(roads['FNODE_'].isin(from_to_set))&(roads['TNODE_'].isin(from_to_set))]
            n=1
        else:
            routes_trace_df=pd.concat([routes_trace_df,roads[(roads['FNODE_'].isin(from_to_set))&(roads['TNODE_'].isin(from_to_set))]])

In [38]:
#可以自行加上点显示，参考folium教程
routes_trace_df.explore()